In [ ]:
!pip install confluent-kafka
!pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 20.9 MB/s eta 0:00:00


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv('.env')

BOOTSTRAP_SERVERS = os.getenv('BOOTSTRAP_SERVERS')
API_KEY = os.getenv('API_KEY')
API_SECRET = os.getenv('API_SECRET')
TOPIC = "incidents_stream"

config = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",
    "sasl.username": API_KEY,
    "sasl.password": API_SECRET,
}


In [4]:
from confluent_kafka import Producer
import json
import random
import time
from datetime import datetime

producer = Producer(config)

alert_codes = ["FLOOD", "FIRE", "ACCIDENT", "PROTEST", "OTHER"]
tags = ["critical", "warning", "info"]

# coordonate aproximative (București)
lat_min, lat_max = 44.38, 44.48
lon_min, lon_max = 26.05, 26.20


In [5]:
def generate_incident():
    alert = random.choice(alert_codes)
    descriptions = {
        "FLOOD": "Inundație în zonă rezidențială.",
        "FIRE": "Incendiu la o clădire.",
        "ACCIDENT": "Accident rutier la intersecție.",
        "PROTEST": "Protest în centrul orașului.",
        "OTHER": "Incident neclasificat raportat.",
    }

    return {
        "id": random.randint(1, 1_000_000),
        "reported_at": datetime.utcnow().isoformat(),
        "lat": round(random.uniform(lat_min, lat_max), 6),
        "lon": round(random.uniform(lon_min, lon_max), 6),
        "alert_code": alert,
        "description": descriptions[alert],
        "tag": random.choice(tags),
    }


In [8]:
print("Starting producer... (CTRL+STOP to halt)")

try:
    while True:
        incident = generate_incident()
        producer.produce(
            TOPIC,
            value=json.dumps(incident).encode("utf-8")
        )
        producer.poll(0)
        print("Sent:", incident)
        time.sleep(3)
except KeyboardInterrupt:
    print("Producer stopped.")
finally:
    producer.flush()


Starting producer... (CTRL+STOP to halt)
Sent: {'id': 402979, 'reported_at': '2025-11-23T17:17:58.310700', 'lat': 44.453667, 'lon': 26.088111, 'alert_code': 'ACCIDENT', 'description': 'Accident rutier la intersecție.', 'tag': 'info'}


/tmp/ipython-input-2116736526.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "reported_at": datetime.utcnow().isoformat(),


Sent: {'id': 490830, 'reported_at': '2025-11-23T17:18:01.311027', 'lat': 44.47699, 'lon': 26.165988, 'alert_code': 'OTHER', 'description': 'Incident neclasificat raportat.', 'tag': 'critical'}
Sent: {'id': 49582, 'reported_at': '2025-11-23T17:18:04.311604', 'lat': 44.422245, 'lon': 26.056089, 'alert_code': 'ACCIDENT', 'description': 'Accident rutier la intersecție.', 'tag': 'critical'}
Sent: {'id': 942398, 'reported_at': '2025-11-23T17:18:07.312077', 'lat': 44.416875, 'lon': 26.189506, 'alert_code': 'PROTEST', 'description': 'Protest în centrul orașului.', 'tag': 'critical'}
Sent: {'id': 741618, 'reported_at': '2025-11-23T17:18:10.313980', 'lat': 44.417248, 'lon': 26.084283, 'alert_code': 'FIRE', 'description': 'Incendiu la o clădire.', 'tag': 'critical'}
Sent: {'id': 300069, 'reported_at': '2025-11-23T17:18:13.314589', 'lat': 44.409608, 'lon': 26.157745, 'alert_code': 'ACCIDENT', 'description': 'Accident rutier la intersecție.', 'tag': 'warning'}
Sent: {'id': 2322, 'reported_at': '202